In [17]:
import torch
import torch.nn as nn
import torch.optim as optim

# 예시 데이터
data = [
    ("경복궁초등학교CU편의점", "편의점"),
    ("강남이디야커피", "카페"),
    ("파리바게뜨강남역", "제과점"),
    ("교촌치킨신촌점", "음식점"),
    ("한성중학교", "학교")
]

# 카테고리 인덱싱
categories = sorted(set([c for _, c in data]))
cat2idx = {c: i for i, c in enumerate(categories)}

# 문자 사전 만들기
chars = sorted(set("".join([name for name, _ in data])))
char2idx = {ch: i+1 for i, ch in enumerate(chars)}  # 0은 padding
vocab_size = len(char2idx) + 1

# 데이터 숫자화
def encode_name(name, max_len=20):
    idxs = [char2idx.get(ch, 0) for ch in name]
    if len(idxs) < max_len:
        idxs += [0] * (max_len - len(idxs))
    return torch.tensor(idxs[:max_len])

X = torch.stack([encode_name(n) for n, _ in data])
y = torch.tensor([cat2idx[c] for _, c in data])

# 모델 정의
class ShopClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embed(x)
        _, (h, _) = self.lstm(x)
        out = self.fc(h[-1])
        return out

model = ShopClassifier(vocab_size, 32, 64, len(categories))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 간단한 학습 루프
for epoch in range(50):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss {loss.item():.4f}")

# 테스트
test = encode_name("고려대학교CU편의점")
pred = model(test.unsqueeze(0))
pred_cat = categories[pred.argmax().item()]
print("예측 결과:", pred_cat)

Epoch 0, Loss 1.6132
Epoch 10, Loss 1.6068
Epoch 20, Loss 1.5785
Epoch 30, Loss 1.3474
Epoch 40, Loss 1.1734
예측 결과: 편의점
